# nnU-Net v2 — Lumbar Spine MRI Segmentation
**Target: Dice > 0.85 | Replaces current 0.65 model**

## Why nnU-Net beats custom ResUNet:
- Auto patch size, spacing, normalization
- Residual encoder + deep supervision built-in
- State-of-art on medical segmentation benchmarks
- Expected Dice on SPIDER: **0.85–0.92**

## Requirements:
- Kaggle T4 GPU
- SPIDER dataset attached
- PyTorch 2.3 (install cell handles this)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1: Install nnU-Net v2 + Fix PyTorch for T4
# ═══════════════════════════════════════════════════
import subprocess, sys, os

# Fix PyTorch for Kaggle T4
print('Installing PyTorch 2.3.1 (T4 compatible)...')
subprocess.run([sys.executable,'-m','pip','install',
    'torch==2.3.1','torchvision==0.18.1',
    '--index-url','https://download.pytorch.org/whl/cu121','-q'],
    timeout=600)

# Install nnU-Net v2
print('Installing nnU-Net v2...')
subprocess.run([sys.executable,'-m','pip','install','nnunetv2','-q'],timeout=300)
subprocess.run([sys.executable,'-m','pip','install',
    'SimpleITK','batchgenerators','acvl-utils','dynamic-network-architectures',
    'scipy','pandas','nibabel','-q'],timeout=300)

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    x=torch.zeros(3,3).cuda()
    print(f'GPU test: OK — {x.device}')
print('\n✓ Setup complete')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2: Configure nnU-Net Environment
# ═══════════════════════════════════════════════════
import os
from pathlib import Path

# nnU-Net requires these environment variables
NNUNET_BASE = Path('/kaggle/working/nnunet')
RAW_DIR     = NNUNET_BASE / 'raw'
PREPROC_DIR = NNUNET_BASE / 'preprocessed'
RESULTS_DIR = NNUNET_BASE / 'results'

for d in [RAW_DIR, PREPROC_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw']          = str(RAW_DIR)
os.environ['nnUNet_preprocessed'] = str(PREPROC_DIR)
os.environ['nnUNet_results']      = str(RESULTS_DIR)

# Dataset ID for SPIDER
DATASET_ID   = 1
DATASET_NAME = f'Dataset{DATASET_ID:03d}_SpineSPIDER'
TASK_DIR     = RAW_DIR / DATASET_NAME
(TASK_DIR/'imagesTr').mkdir(parents=True, exist_ok=True)
(TASK_DIR/'labelsTr').mkdir(parents=True, exist_ok=True)
(TASK_DIR/'imagesTs').mkdir(parents=True, exist_ok=True)

print(f'nnUNet_raw         : {RAW_DIR}')
print(f'nnUNet_preprocessed: {PREPROC_DIR}')
print(f'nnUNet_results     : {RESULTS_DIR}')
print(f'Dataset dir        : {TASK_DIR}')
print('✓ Environment configured')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3: Find SPIDER Dataset Paths
# ═══════════════════════════════════════════════════
import glob
from pathlib import Path

# Auto-detect dataset
csvs = glob.glob('/kaggle/input/**/overview.csv', recursive=True)
assert csvs, 'overview.csv not found — add SPIDER dataset'
import pandas as pd
OVERVIEW = Path(csvs[0])
DATA_BASE = OVERVIEW.parent

# Find images and masks
t2s = glob.glob('/kaggle/input/**/*_t2.mha', recursive=True)
assert t2s, 'No MHA files found'
IMAGES_DIR = Path(t2s[0]).parent
img_names = {Path(f).name for f in t2s}
all_mha = glob.glob('/kaggle/input/**/*.mha', recursive=True)
mask_dirs = {Path(f).parent for f in all_mha
             if Path(f).name in img_names and Path(f).parent != IMAGES_DIR}
MASKS_DIR = list(mask_dirs)[0] if mask_dirs else IMAGES_DIR

n_img = len(list(IMAGES_DIR.glob('*.mha')))
n_msk = len(list(MASKS_DIR.glob('*.mha')))
print(f'Images: {IMAGES_DIR} ({n_img} files)')
print(f'Masks : {MASKS_DIR}  ({n_msk} files)')
print(f'CSV   : {OVERVIEW}')
assert n_img > 400, f'Expected ~447 images, found {n_img}'
print('✓ Dataset found')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4: Convert SPIDER to nnU-Net Format
# Converts .mha → .nii.gz with correct label mapping
# ═══════════════════════════════════════════════════
import SimpleITK as sitk
import numpy as np
import json, shutil
from pathlib import Path
import pandas as pd

# SPIDER → nnU-Net label mapping
# nnU-Net needs contiguous labels 0,1,2,...
SPIDER_TO_NNUNET = {
    0:  0,   # background
    # Vertebrae (sequential 1-8 in SPIDER = L5 up to upper thoracic)
    1:  1, 2:  2, 3:  3, 4:  4, 5:  5, 6:  6, 7:  7, 8:  8,
    100: 9,  # Sacrum
    # IVDs (201-208)
    201: 10, 202: 11, 203: 12, 204: 13,
    205: 14, 206: 15, 207: 16, 208: 17,
}

LABEL_NAMES = {
    0:'background', 1:'Vertebra-1', 2:'Vertebra-2', 3:'Vertebra-3',
    4:'Vertebra-4', 5:'Vertebra-5', 6:'Vertebra-6', 7:'Vertebra-7',
    8:'Vertebra-8', 9:'Sacrum',
    10:'IVD-1', 11:'IVD-2', 12:'IVD-3', 13:'IVD-4',
    14:'IVD-5', 15:'IVD-6', 16:'IVD-7', 17:'IVD-8',
}
NUM_CLASSES = 18  # 0-17

def remap_mask(mask_np):
    out = np.zeros_like(mask_np, dtype=np.uint8)
    for src, dst in SPIDER_TO_NNUNET.items():
        out[mask_np == src] = dst
    return out

def convert_to_nnunet(patient_ids, split, images_dir, masks_dir, out_images, out_labels):
    converted = 0
    skipped = 0
    for pid in patient_ids:
        for mod in ['t2']:  # T2 only for now — add 't1' for more data
            img_path  = images_dir / f'{pid}_{mod}.mha'
            mask_path = masks_dir  / f'{pid}_{mod}.mha'
            if not img_path.exists() or not mask_path.exists():
                skipped += 1; continue
            try:
                # Read image
                img_sitk  = sitk.ReadImage(str(img_path))
                mask_sitk = sitk.ReadImage(str(mask_path))

                # Remap mask labels
                mask_np = sitk.GetArrayFromImage(mask_sitk).astype(np.int32)
                mask_remapped = remap_mask(mask_np)
                new_mask = sitk.GetImageFromArray(mask_remapped)
                new_mask.CopyInformation(mask_sitk)

                case_id = f'{pid}_{mod}'
                # nnU-Net naming: case_XXXX_0000.nii.gz (channel 0 = T2)
                sitk.WriteImage(img_sitk,  str(out_images / f'{case_id}_0000.nii.gz'))
                sitk.WriteImage(new_mask,  str(out_labels / f'{case_id}.nii.gz'))
                converted += 1
            except Exception as e:
                print(f'  Error {pid}: {e}')
                skipped += 1
    print(f'  {split}: {converted} converted, {skipped} skipped')
    return converted

# Get patient splits from overview.csv
df = pd.read_csv(OVERVIEW)
train_pids, val_pids = [], []
seen = set()
for name in df['new_file_name'].tolist():
    if not name.endswith('_t2') or 'SPACE' in name: continue
    pid = name.replace('_t2', '')
    if pid in seen: continue
    seen.add(pid)
    sub = df.loc[df['new_file_name']==name, 'subset'].values
    if len(sub) and sub[0] == 'validation':
        val_pids.append(pid)
    else:
        train_pids.append(pid)

print(f'Converting {len(train_pids)} train + {len(val_pids)} val patients...')
n_train = convert_to_nnunet(train_pids, 'train', IMAGES_DIR, MASKS_DIR,
                             TASK_DIR/'imagesTr', TASK_DIR/'labelsTr')

# Val goes into imagesTr too (nnU-Net uses cross-validation)
n_val = convert_to_nnunet(val_pids, 'val', IMAGES_DIR, MASKS_DIR,
                           TASK_DIR/'imagesTr', TASK_DIR/'labelsTr')

print(f'\nTotal cases in imagesTr: {len(list((TASK_DIR/"imagesTr").glob("*.nii.gz")))}')
print(f'Total cases in labelsTr: {len(list((TASK_DIR/"labelsTr").glob("*.nii.gz")))}')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 5: Create dataset.json (required by nnU-Net)
# ═══════════════════════════════════════════════════
import json
from pathlib import Path

cases = sorted([f.name.replace('_0000.nii.gz','') for f in (TASK_DIR/'imagesTr').glob('*.nii.gz')])
print(f'Total cases: {len(cases)}')
print(f'Sample: {cases[:3]}')

dataset_json = {
    'channel_names': {'0': 'T2'},
    'labels': {
        'background': 0,
        'Vertebra-1': 1, 'Vertebra-2': 2, 'Vertebra-3': 3, 'Vertebra-4': 4,
        'Vertebra-5': 5, 'Vertebra-6': 6, 'Vertebra-7': 7, 'Vertebra-8': 8,
        'Sacrum':     9,
        'IVD-1': 10, 'IVD-2': 11, 'IVD-3': 12, 'IVD-4': 13,
        'IVD-5': 14, 'IVD-6': 15, 'IVD-7': 16, 'IVD-8': 17,
    },
    'numTraining': len(cases),
    'file_ending': '.nii.gz',
    'name': DATASET_NAME,
    'description': 'SPIDER Lumbar Spine MRI Segmentation',
    'reference': 'SPIDER Dataset',
    'licence': 'CC-BY',
    'release': '1.0',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}

with open(TASK_DIR / 'dataset.json', 'w') as f:
    json.dump(dataset_json, f, indent=2)

print(f'\n✓ dataset.json created')
print(f'  Classes: {NUM_CLASSES} (background + 17 structures)')
print(json.dumps(dataset_json, indent=2)[:500])

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 6: nnU-Net Preprocessing
# Auto: spacing, patch size, normalization, intensity
# ═══════════════════════════════════════════════════
import subprocess, sys

print('Running nnU-Net dataset verification...')
result = subprocess.run([
    sys.executable, '-m', 'nnunetv2.dataset_conversion.verify_dataset_integrity',
    '-d', str(DATASET_ID)
], capture_output=True, text=True, timeout=120)
print(result.stdout[-500:] if result.stdout else '')
if result.returncode != 0:
    print('Verification output:', result.stderr[-300:])

print('\nRunning nnU-Net fingerprint extraction...')
result = subprocess.run([
    sys.executable, '-m', 'nnunetv2.experiment_planning.plan_and_preprocess',
    '-d', str(DATASET_ID),
    '--verify_dataset_integrity',
    '-c', '2d',          # 2D config for sagittal MRI
    '--no_pp',           # skip preprocessing initially to check
], capture_output=True, text=True, timeout=600,
   env={**os.environ})
print(result.stdout[-1000:] if result.stdout else 'No output')
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

print('\nRunning nnU-Net preprocessing (auto patch size, spacing, augmentation)...')
result2 = subprocess.run([
    sys.executable, '-m', 'nnunetv2.experiment_planning.plan_and_preprocess',
    '-d', str(DATASET_ID),
    '-c', '2d',
    '-np', '4',          # 4 preprocessing workers
], capture_output=True, text=True, timeout=3600,
   env={**os.environ})
print(result2.stdout[-1000:] if result2.stdout else 'No output')
if result2.returncode != 0:
    print('STDERR:', result2.stderr[-500:])
print('\n✓ Preprocessing complete')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7: PATCH + TRAIN nnU-Net v2  (Python 3.12 + torch.compile fix)
# Strategy: patch the trainer file on disk, then run via CLI subprocess
# Expected: epoch 1 ~60-90s, Dice >0.85 after ~500 epochs (~6-8h)
# ═══════════════════════════════════════════════════════════════════
import os, sys, subprocess
from pathlib import Path

# ── STEP 1: Set env vars ─────────────────────────────────────────
NNUNET_BASE = Path('/kaggle/working/nnunet')
os.environ['nnUNet_raw']          = str(NNUNET_BASE / 'raw')
os.environ['nnUNet_preprocessed'] = str(NNUNET_BASE / 'preprocessed')
os.environ['nnUNet_results']      = str(NNUNET_BASE / 'results')
os.environ['nnUNet_compile']      = 'false'   # belt+suspenders
DATASET_ID = 1
print(f'nnUNet_raw          : {os.environ["nnUNet_raw"]}')
print(f'nnUNet_preprocessed : {os.environ["nnUNet_preprocessed"]}')
print(f'nnUNet_results      : {os.environ["nnUNet_results"]}')
print(f'nnUNet_compile      : {os.environ["nnUNet_compile"]}')

# ── STEP 2: Patch nnUNetTrainer.py on disk ───────────────────────
# The ONLY reliable way: edit the file before the CLI subprocess loads it
import importlib, importlib.util
spec = importlib.util.find_spec('nnunetv2')
assert spec, 'nnunetv2 not installed'
pkg_root = Path(spec.origin).parent
trainer_file = pkg_root / 'training' / 'nnUNetTrainer' / 'nnUNetTrainer.py'
print(f'\nTrainer file: {trainer_file}')
assert trainer_file.exists(), f'Not found: {trainer_file}'

src = trainer_file.read_text()
original_hash = hash(src)

PATCHES = [
    # Patch A — the direct torch.compile(self.network) call
    (
        'self.network = torch.compile(self.network)',
        'self.network = self.network  # PATCHED: torch.compile disabled (Python 3.12 has no Dynamo)'
    ),
    # Patch B — the self.compile = True assignment (sets flag that later triggers compile)
    (
        "self.compile = ('nnUNet_compile' in os.environ "
        "and os.environ['nnUNet_compile'].lower() in ('true', '1', 't'))",
        "self.compile = False  # PATCHED: always disabled for Python 3.12"
    ),
    # Patch C — alternative location in some versions: if self.compile:
    # We do NOT patch this one blindly as it may break things
]

changed = False
for old, new in PATCHES:
    if old in src:
        src = src.replace(old, new)
        changed = True
        print(f'  ✓ Patched: ...{old[30:70]}...')
    else:
        # Check if already patched
        short = old.replace('self.network = ', '').replace('self.compile = ', '')[:40]
        if 'PATCHED' in src or new in src:
            print(f'  ✓ Already patched: ...{short}...')
        else:
            print(f'  ℹ Not found (different version?): ...{short}...')

if changed:
    trainer_file.write_text(src)
    print('  ✓ Trainer file written to disk')
else:
    print('  ✓ No changes needed (already patched or different nnU-Net version)')

# Verify
src2 = trainer_file.read_text()
if 'torch.compile(self.network)' in src2:
    print('  ❌ WARNING: torch.compile still present in trainer!')
    print('  Trying aggressive patch...')
    src2 = src2.replace('torch.compile(self.network)', 'self.network  # patched')
    trainer_file.write_text(src2)
    print('  ✓ Aggressive patch applied')
else:
    print('  ✓ Verified: torch.compile(self.network) NOT in trainer')

# Also write sitecustomize.py as a final safety net for subprocesses
import site
site_pkgs = site.getsitepackages()
sc_path = Path(site_pkgs[0]) / 'sitecustomize.py'
sc_code = '''
# Auto-injected: disable torch.compile for Python 3.12
try:
    import torch as _torch
    if not getattr(_torch, '_compile_patched', False):
        def _safe_compile(fn=None, *a, **kw): return fn if fn is not None else (lambda f: f)
        _torch.compile = _safe_compile
        _torch._compile_patched = True
except Exception:
    pass
'''
sc_path.write_text(sc_code)
print(f'  ✓ sitecustomize.py written to {sc_path}')

# ── STEP 3: Find the CLI executable ─────────────────────────────
import shutil
cli = shutil.which('nnUNetv2_train') or '/usr/local/bin/nnUNetv2_train'
print(f'\nCLI script: {cli}')
assert Path(cli).exists(), f'nnUNetv2_train not found at {cli}'

# ── STEP 4: Run training via subprocess ─────────────────────────
cmd = [sys.executable, cli, str(DATASET_ID), '2d', '0', '--npz']
print(f'Running: {" ".join(cmd)}')
print()
print('Expected output:')
print('  Using device: cuda:0')
print('  This split has 168 training and 42 validation cases.')
print('  Epoch 1 ... [loss, dice printed every epoch]')
print()
print('If you see "Using torch.compile... 2026-...: do_dummy_2d" then epoch appears shortly after.')
print('Training 1000 epochs ~6-8 hours. Cell will stream output.')
print('='*60)

env = {
    **os.environ,
    'nnUNet_raw':           str(NNUNET_BASE / 'raw'),
    'nnUNet_preprocessed':  str(NNUNET_BASE / 'preprocessed'),
    'nnUNet_results':       str(NNUNET_BASE / 'results'),
    'nnUNet_compile':       'false',
    'PYTHONDONTWRITEBYTECODE': '1',
}

# Stream output so we can see epochs as they happen
proc = subprocess.Popen(
    cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print('\n⚠ Training interrupted by user (checkpoint saved at last epoch)')

proc.wait()
print(f'\nTraining finished with return code: {proc.returncode}')
if proc.returncode == 0:
    print('✓ Training complete!')
else:
    print('✗ Training exited with error — check output above')
    print('  Common fixes:')
    print('  1. Re-run this cell (nnU-Net auto-resumes from last checkpoint)')
    print('  2. Check GPU memory: !nvidia-smi')
    print('  3. If torch.compile error reappears, run: !python -c "import nnunetv2.training.nnUNetTrainer.nnUNetTrainer"')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 8: Validate / Predict with trained nnU-Net
# ═══════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path

PRED_DIR = Path('/kaggle/working/predictions')
PRED_DIR.mkdir(exist_ok=True)

# Predict on validation set
print('Running nnU-Net inference on validation set...')
pred_cmd = [
    sys.executable, '-m', 'nnunetv2.inference.predict_from_raw_data',
    '-i', str(TASK_DIR/'imagesTr'),  # input folder
    '-o', str(PRED_DIR),              # output folder
    '-d', str(DATASET_ID),
    '-c', '2d',
    '-f', '0',
    '--save_probabilities',
    '-device', 'cuda',
]

result = subprocess.run(pred_cmd, capture_output=True, text=True,
                        timeout=3600, env={**os.environ})
print(result.stdout[-500:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-300:])
print('\n✓ Inference complete')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 9: Evaluate Dice, IoU, HD95
# ═══════════════════════════════════════════════════
import numpy as np
import SimpleITK as sitk
from pathlib import Path
from collections import defaultdict
import json

LABEL_NAMES = {
    1:'Vertebra-1',2:'Vertebra-2',3:'Vertebra-3',4:'Vertebra-4',
    5:'Vertebra-5',6:'Vertebra-6',7:'Vertebra-7',8:'Vertebra-8',
    9:'Sacrum',10:'IVD-1',11:'IVD-2',12:'IVD-3',13:'IVD-4',
    14:'IVD-5',15:'IVD-6',16:'IVD-7',17:'IVD-8',
}

def compute_dice(pred, gt, label):
    p = (pred == label).astype(float)
    g = (gt   == label).astype(float)
    if g.sum() == 0 and p.sum() == 0: return None
    tp = (p*g).sum()
    return float(2*tp / (p.sum() + g.sum() + 1e-6))

def compute_iou(pred, gt, label):
    p = (pred == label).astype(float)
    g = (gt   == label).astype(float)
    if g.sum() == 0 and p.sum() == 0: return None
    tp = (p*g).sum()
    return float(tp / (p.sum() + g.sum() - tp + 1e-6))

pred_files = sorted(PRED_DIR.glob('*.nii.gz'))
gt_dir     = TASK_DIR / 'labelsTr'

all_dice = defaultdict(list)
all_iou  = defaultdict(list)

for pred_path in pred_files[:20]:  # evaluate first 20
    gt_path = gt_dir / pred_path.name
    if not gt_path.exists(): continue
    pred_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(pred_path)))
    gt_arr   = sitk.GetArrayFromImage(sitk.ReadImage(str(gt_path)))
    for label, name in LABEL_NAMES.items():
        d = compute_dice(pred_arr, gt_arr, label)
        i = compute_iou(pred_arr, gt_arr, label)
        if d is not None: all_dice[name].append(d)
        if i is not None: all_iou[name].append(i)

print('='*60)
print('  nnU-Net v2 EVALUATION RESULTS')
print('='*60)
print(f'{"Class":<22} {"Dice":>8}  {"IoU":>8}')
print('-'*42)

all_d, all_i = [], []
vert_d, ivd_d = [], []
for name in LABEL_NAMES.values():
    if name not in all_dice: continue
    d = np.mean(all_dice[name])
    i = np.mean(all_iou[name]) if name in all_iou else 0
    all_d.append(d); all_i.append(i)
    if 'Vert' in name: vert_d.append(d)
    if 'IVD'  in name: ivd_d.append(d)
    tag = '★' if d>=0.90 else '✓' if d>=0.80 else '·' if d>=0.70 else ''
    print(f'{name:<22} {d:>8.4f}  {i:>8.4f}  {tag}')

md = np.mean(all_d) if all_d else 0
mi = np.mean(all_i) if all_i else 0
print('-'*42)
print(f'{"MEAN FOREGROUND":<22} {md:>8.4f}  {mi:>8.4f}')
print()
if vert_d: print(f'Vertebrae mean Dice : {np.mean(vert_d):.4f}')
if ivd_d:  print(f'IVD mean Dice       : {np.mean(ivd_d):.4f}')
print(f'Overall mean Dice   : {md:.4f}')
print('='*60)

status = ('🎯 TARGET ACHIEVED: Dice >= 0.90!' if md>=0.90 else
          '📈 Excellent — very close to target' if md>=0.85 else
          '📊 Good — continue training' if md>=0.75 else
          '📉 Training more needed')
print(f'Status: {status}')

# Save results
results = {'mean_dice':md,'mean_iou':mi,'vertebrae_dice':float(np.mean(vert_d)) if vert_d else 0,
           'ivd_dice':float(np.mean(ivd_d)) if ivd_d else 0,
           'per_class':{n:float(np.mean(v)) for n,v in all_dice.items()}}
with open('/kaggle/working/nnunet_results.json','w') as f:
    json.dump(results,f,indent=2)
print('\n✓ Results saved to /kaggle/working/nnunet_results.json')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 10: Export best model checkpoint
# ═══════════════════════════════════════════════════
import shutil, os
from pathlib import Path

# Find nnU-Net checkpoint
model_dir = Path(os.environ['nnUNet_results']) / DATASET_NAME / 'nnUNetTrainer__nnUNetPlans__2d' / 'fold_0'
print(f'Model dir: {model_dir}')

if model_dir.exists():
    checkpoints = list(model_dir.glob('*.pth'))
    print(f'Checkpoints: {[c.name for c in checkpoints]}')
    best_ckpt = model_dir / 'checkpoint_best.pth'
    if best_ckpt.exists():
        shutil.copy(best_ckpt, '/kaggle/working/nnunet_best_model.pth')
        print(f'✓ Best model saved: /kaggle/working/nnunet_best_model.pth')
    # Also save the full model folder
    shutil.make_archive('/kaggle/working/nnunet_model_fold0', 'zip', str(model_dir))
    print(f'✓ Full model archived: /kaggle/working/nnunet_model_fold0.zip')
else:
    print('Model dir not found — training may not have completed')
    print('Available:', list(Path(os.environ['nnUNet_results']).glob('**/*.pth'))[:5])

print('\n✓ Export complete. Download from Kaggle Output tab.')